In [ ]:
import sys
import os
import subprocess
from pathlib import Path

# Ensure project import path
PROJECT_ROOT = Path('/global/home/hpc5656/SLAM')
sys.path.append(str(PROJECT_ROOT))
# Set working directory so relative paths (e.g., src/config/*.yaml) resolve
os.chdir(str(PROJECT_ROOT))
print('CWD:', Path.cwd())

%matplotlib inline

# CuPy is required - ensure CUDA_PATH and LD_LIBRARY_PATH are set
# This allows the notebook to work even if Jupyter wasn't started with modules loaded
if "CUDA_PATH" not in os.environ:
    print("CUDA_PATH not set, attempting to load modules...")
    try:
        result = subprocess.run(
            'module load cuda/12.2 && env',
            shell=True,
            executable='/bin/bash',
            capture_output=True,
            text=True,
            timeout=10
        )
        if result.returncode == 0:
            for line in result.stdout.split('\n'):
                if '=' in line:
                    key, value = line.split('=', 1)
                    os.environ[key] = value
            print(f"✓ Modules loaded. CUDA_PATH: {os.environ.get('CUDA_PATH', 'Not set')}")
        else:
            print(f"⚠ Could not load modules. Error: {result.stderr}")
            raise RuntimeError(
                "CUDA_PATH not set and could not load modules. "
                "Please run: module load cuda/12.2 before starting Jupyter."
            )
    except Exception as e:
        print(f"✗ Could not load modules: {e}")
        raise RuntimeError(
            f"Failed to load CUDA modules: {e}\n"
            "Please ensure CUDA is loaded before starting Jupyter:\n"
            "  module load cuda/12.2"
        ) from e

# Ensure LD_LIBRARY_PATH includes CUDA library directory for NVRTC (libnvrtc.so.12)
cuda_path = os.environ.get('CUDA_PATH')
libnvrtc_path = None

if cuda_path:
    cuda_lib_paths = [
        os.path.join(cuda_path, 'lib64'),
        os.path.join(cuda_path, 'targets', 'x86_64-linux', 'lib'),
    ]
    
    current_ld_path = os.environ.get('LD_LIBRARY_PATH', '')
    ld_paths = current_ld_path.split(':') if current_ld_path else []
    paths_added = []
    
    for cuda_lib_path in cuda_lib_paths:
        if os.path.exists(cuda_lib_path):
            if cuda_lib_path not in ld_paths:
                paths_added.append(cuda_lib_path)
                ld_paths.insert(0, cuda_lib_path)
    
    if paths_added:
        os.environ['LD_LIBRARY_PATH'] = ':'.join(ld_paths)
        print(f"✓ Updated LD_LIBRARY_PATH to include: {', '.join(paths_added)}")
    
    for cuda_lib_path in cuda_lib_paths:
        if os.path.exists(cuda_lib_path):
            potential_libnvrtc = os.path.join(cuda_lib_path, 'libnvrtc.so.12')
            if os.path.exists(potential_libnvrtc):
                libnvrtc_path = potential_libnvrtc
                print(f"✓ Found libnvrtc.so.12 at: {libnvrtc_path}")
                try:
                    import ctypes
                    try:
                        lib = ctypes.CDLL(libnvrtc_path, mode=ctypes.RTLD_GLOBAL)
                        print(f"✓ Preloaded libnvrtc.so.12 using ctypes (RTLD_GLOBAL)")
                    except Exception as e1:
                        try:
                            lib = ctypes.CDLL(libnvrtc_path)
                            print(f"✓ Preloaded libnvrtc.so.12 using ctypes (standard)")
                        except Exception as e2:
                            raise e1 from e2
                except Exception as e:
                    print(f"⚠ Warning: Could not preload libnvrtc.so.12: {e}")
                
                try:
                    import ctypes.util
                    found_lib = ctypes.util.find_library('nvrtc')
                    if found_lib:
                        print(f"✓ ctypes.util.find_library('nvrtc') found: {found_lib}")
                except Exception:
                    pass
                break

from src.utils.array_backend import np, random, is_cupy
from src.classes.belief_mdp_n_M import BeliefMDP_n_M_SLAM
from src.classes.mapping import OrderedLandmarkMap
from src.classes.model import RangeBearingSensor, DoubleIntegratorModel
from tqdm import tqdm
import time
import warnings

# Suppress CuPy experimental FutureWarnings
warnings.filterwarnings('ignore', category=FutureWarning, module='cupy.random')

print(f"✓ All imports successful")
print(f"Using backend: {'CuPy (GPU)' if is_cupy else 'NumPy (CPU)'}")

# Verify we're using CuPy
if not is_cupy:
    raise RuntimeError(
        "CuPy is required but not being used. "
        "Check CUDA installation and CuPy setup."
    )

"""
Generate c_n^{(M)} cost matrix for BeliefMDP_n_M_SLAM (landmark SLAM).

This notebook:
1. Creates a BeliefMDP_n_M_SLAM instance with coarse quantization (n=2)
2. Loads or computes p_n^{(M)} and c_n^{(M)}
3. Verifies the computed cost matrix properties

For SLAM (landmark + range-bearing):
- Belief space: π(x, m) over state and landmark map hypotheses
- Quantized belief space: Π_n^(M)
- c_n^{(M)} shape: (cardinality, m_n, n_u)
"""

In [ ]:
# Configuration
n = 2  # Coarse quantization for initial testing
M = 3  # Belief space quantization parameter
beta = 0.95  # Discount factor
obs_n = n + 1  # Observation quantization
action_n = 4  # Action quantization

print(f"Configuration:")
print(f"  n (state quantization): {n}")
print(f"  M (belief quantization): {M}")
print(f"  β (discount factor): {beta}")
print(f"  obs_n (observation quantization): {obs_n}")
print(f"  action_n (action quantization): {action_n}")
print()

# Landmark map setup (ordered landmark tuples)
workspace_min = 0.0
workspace_max = 10.0
num_landmarks = 3

cell_size = (workspace_max - workspace_min) / n
coords = workspace_min + cell_size * (np.arange(n) + 0.5)
xx, yy = np.meshgrid(coords, coords)
landmark_positions = np.stack([xx.ravel(), yy.ravel()], axis=1)

landmark_map = OrderedLandmarkMap(
    x_min=workspace_min,
    x_max=workspace_max,
    y_min=workspace_min,
    y_max=workspace_max,
    landmark_positions=landmark_positions,
    num_landmarks=num_landmarks,
)

# Create models
motion_model = DoubleIntegratorModel(
    p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=1.0, max_a=1.0
)
sensor = RangeBearingSensor(r_max=6.0, epsilon=0.1, sigma_r=0.05, sigma_phi=0.05)

print(f"Environment setup:")
print(f"  Map bounds: x=[{workspace_min}, {workspace_max}], y=[{workspace_min}, {workspace_max}]")
print(f"  Landmark grid: {n}x{n} with {num_landmarks} landmarks")
print(f"  Total maps: {landmark_map.len_M}")
print(f"  Motion model: DoubleIntegratorModel (dt={motion_model.dt}, max_a={motion_model.max_a})")
print(f"  Sensor: RangeBearingSensor (r_max={sensor.r_max}, eps={sensor.epsilon})")

In [ ]:
# Create BeliefMDP_n_M_SLAM instance
print("="*70)
print("Creating BeliefMDP_n_M_SLAM instance...")
print("="*70)
print()

start_time = time.time()

bmdp_M = BeliefMDP_n_M_SLAM(
    M=M,
    β=beta,
    n=n,
    motion_model=motion_model,
    measurement_model=sensor,
    obstacles=[],
    _map=landmark_map,
    sigma_w=0.01,
    sigma_v=0.01,
    exploration_type="information gain",
    obs_n=obs_n,
    action_n=action_n,
)

elapsed_time = time.time() - start_time

print()
print("="*70)
print("Initialization complete!")
print("="*70)
print(f"Total time: {elapsed_time:.2f}s")
print()
print(f"Belief space statistics:")
print(f"  Cardinality |Π_n^M|: {bmdp_M.BQ.cardinality:,}")
print(f"  State space size m_n: {bmdp_M.SQ.m_n}")
print(f"  Map hypothesis count len_M: {bmdp_M.len_M}")
print(f"  Action space size n_u: {bmdp_M.AQ.n_u}")
print(f"  c_n^{(M)} shape: {bmdp_M.c_n_M.shape}")
print(f"  c_n^{(M)} dtype: {bmdp_M.c_n_M.dtype}")

In [ ]:
# Verify c_n^{(M)} properties
print("="*70)
print("Verifying c_n^{(M)} properties...")
print("="*70)
print()

c_n_M = bmdp_M.c_n_M
cardinality, m_n, n_u = c_n_M.shape

print(f"Shape: ({cardinality}, {m_n}, {n_u})")
print(f"  - {cardinality:,} beliefs")
print(f"  - {m_n} states")
print(f"  - {n_u} actions")
print(f"  - Total entries: {c_n_M.size:,}")
print()

# Value statistics
min_val = float(np.min(c_n_M))
max_val = float(np.max(c_n_M))
mean_val = float(np.mean(c_n_M))
std_val = float(np.std(c_n_M))

print(f"Value statistics:")
print(f"  Min: {min_val:.6f}")
print(f"  Max: {max_val:.6f}")
print(f"  Mean: {mean_val:.6f}")
print(f"  Std: {std_val:.6f}")
print()

if min_val < 0:
    print(f"  ⚠ WARNING: Found negative values!")
else:
    print(f"  ✓ All values are non-negative")
print()

# Check for NaN or Inf
has_nan = bool(np.any(np.isnan(c_n_M)))
has_inf = bool(np.any(np.isinf(c_n_M)))

if has_nan:
    print(f"  ⚠ WARNING: Found NaN values!")
else:
    print(f"  ✓ No NaN values")

if has_inf:
    print(f"  ⚠ WARNING: Found Inf values!")
else:
    print(f"  ✓ No Inf values")
print()

print("="*70)
print("Verification complete!")
print("="*70)